# Fine-Tuning BERT for Presidential Motive Imagery**Deep Learning for Text and Vision — BERT lab**---In the 1970s and 80s, David Winter and colleagues built a way to measure the *motives* ofpolitical leaders from nothing but their words. The method is content analysis of runningtext: a trained human coder reads a speech sentence by sentence and marks each one for**achievement**, **affiliation-intimacy**, or **power** imagery. Winter scored every U.S.presidential inaugural address this way, and the resulting motive profiles have been used topredict everything from cabinet turnover to entry into war.Coding is slow. Certifying a single human coder takes weeks. So the question this lab asks isthe obvious one:> **Can we fine-tune BERT to reproduce Winter's coding?**### Why this task fits BERT so wellMost social-science text is too long for BERT. A parliamentary speech runs to thousands ofwords; BERT stops at 512 tokens, and you end up chunking, truncating, or apologising.Motive imagery has no such problem. **Winter's unit of analysis is the sentence**, and asentence in a presidential address is about 25 words — roughly 35 BERT tokens. Everythingfits, comfortably, with room to spare. We will see this in §1.4, and it is the reason thislab trains in minutes rather than hours.### The two parts| Part | Method | Labelled data needed ||---|---|---|| **1** (§3) | Zero-shot classification with an NLI model | **none** || **2** (§4–6) | Fine-tuning `bert-base-uncased` on Winter's coded sentences | ~1,400 sentences |We then ask which presidents resemble each other (§6), and finish by validating the modelagainst Winter's own published presidential scores (§7) — a far more demanding test than aheld-out F1.### Winter's three motives| Motive | Coded when the text shows… ||---|---|| **Achievement** | concern with a standard of excellence; doing something well, better, or uniquely; long-term goals || **Affiliation-Intimacy** | warm, friendly or intimate feeling toward others; companionship; nurturant acts; sadness at separation || **Power** | impact on others: forceful action, influence and persuasion, control and regulation, unsolicited help, concern with reputation, arousing strong emotion in others |A sentence can carry **more than one** motive, or none. That makes this a *multi-label*problem, not a multi-class one — a distinction that matters for the loss function, and thatwe come back to in §4.2.

---## 0. SetupColab already has `torch` and `transformers`. We add `striprtf` (only if you bring your own`.rtf` speeches) and download NLTK's sentence splitter and inaugural-address corpus.**Turn on the GPU**: *Runtime → Change runtime type → T4 GPU*. Everything below runs on CPUtoo, but fine-tuning goes from about a minute to about ten.

In [ ]:
%pip -q install transformers torch striprtf nltk scikit-learn matplotlib pandas

In [ ]:
import re, warningsfrom pathlib import Pathimport numpy as npimport pandas as pdimport torchimport matplotlib.pyplot as pltimport nltknltk.download("punkt_tab", quiet=True)   # sentence boundary detectornltk.download("inaugural", quiet=True)   # 59 inaugural addresses, 1789-2021from nltk.tokenize import sent_tokenizefrom transformers import (AutoTokenizer, AutoModel,                          AutoModelForSequenceClassification, pipeline)from tqdm.auto import tqdmwarnings.filterwarnings("ignore")np.random.seed(42)torch.manual_seed(42)rng = np.random.default_rng(42)          # used from §3 onwardDEVICE = "cuda" if torch.cuda.is_available() else "cpu"print(f"Device: {DEVICE}")MOTIVES = ["achievement", "affiliation", "power"]

---## 1. The corpus: inaugural addresses### 1.1 Where the text comes fromNLTK ships all U.S. inaugural addresses from Washington 1789 to Biden 2021 as a built-incorpus. No downloads to manage, no Drive to mount, no `.rtf` to strip — one line and you havetwo and a half centuries of the genre Winter studied.If you have additional speeches as files (Trump 2025, say), §1.3 shows how to add them.

In [ ]:
from nltk.corpus import inauguralfileids = inaugural.fileids()print(f"{len(fileids)} inaugural addresses: {fileids[0]} ... {fileids[-1]}")def parse_fileid(fid):    "'2021-Biden.txt' -> (2021, 'Biden')"    stem = fid.replace(".txt", "")    year, name = stem.split("-", 1)    return int(year), namespeeches_raw = {}for fid in fileids:    year, name = parse_fileid(fid)    speeches_raw[f"{name} {year}"] = inaugural.raw(fid)print(f"\nLoaded {len(speeches_raw)} speeches.")print("Most recent five:", list(speeches_raw)[-5:])

### 1.2 Preprocessing**Less is more with transformers.** No lowercasing, no stopword removal, no stemming — BERTwas pretrained on natural running text and uses capitalisation, function words, andpunctuation as signal. All we do is strip transcript artifacts (`[Applause]`), normalisewhitespace, and split into sentences.Sentence splitting is the important step: it is both what BERT needs and what *Winter'smethod* prescribes.---#### Reading the code: regular expressionsThe cleaning below uses **regular expressions** — a miniature language for describing textpatterns. `re.sub(pattern, replacement, text)` finds every match of `pattern` and replaces it.The `r"..."` prefix marks a *raw string*, which stops Python from eating the backslashes beforethe regex engine sees them. Always write regexes as raw strings.Here is every piece of syntax used anywhere in this notebook:| Syntax | Means | Example ||---|---|---|| `\s` | any whitespace character (space, tab, newline) | `\s` matches the gap in `"a b"` || `+` | *one or more* of the thing before it | `\s+` matches `" "` and also `"   \n  "` || `\[` `\]` | a **literal** square bracket — the backslash removes its special meaning | `\[` matches the `[` in `"[Applause]"` || `.` | any single character | `.` matches `A`, `7`, or a space || `*` | *zero or more* of the thing before it | `.*` matches any run of characters || `?` after `*` | **lazy**: match as *few* characters as possible | see below || `[...]` | a character *class* — match any one character listed | `[abc]` matches `a`, `b`, or `c` || `[^...]` | a *negated* class — match any character **not** listed | `[^a-z]` matches anything that is not a lowercase letter || `a-z` inside `[]` | a range | `[a-z0-9]` = letters and digits |**Why the `?` in `\[.*?\]` matters.** Applied to `"[Applause] and [Laughter]"`:- `\[.*\]` is *greedy* — it matches from the first `[` all the way to the **last** `]`,  swallowing `" and "` in between. One match, too much deleted.- `\[.*?\]` is *lazy* — it stops at the **first** `]` it can. Two separate matches,  `"[Applause]"` and `"[Laughter]"`, and the text between them survives.Getting this wrong silently deletes real speech text, which is exactly the kind ofpreprocessing bug that never shows up until a reviewer asks.#### Reading the code: f-stringsThe `print` statements use f-strings with **format specifiers** after a colon. The grammar is`{value:[alignment][width][,][.precision][type]}`:| Spec | Means | `x = 1234.5678` renders as ||---|---|---|| `{x:.0f}` | fixed-point, 0 decimal places | `1235` || `{x:.3f}` | fixed-point, 3 decimals | `1234.568` || `{x:+.3f}` | same, but always show the sign | `+1234.568` || `{x:,}` | thousands separators | `1,234.5678` || `{x:.1%}` | as a percentage, 1 decimal | `123456.8%` || `{s:<12}` | **left**-align in a 12-character column | `` `abc         ` `` || `{s:>5}` | **right**-align in a 5-character column | `` `  abc` `` || `{s!r}` | use `repr(s)` — shows quotes and escapes | `'abc'` instead of `abc` |So the line in §3.1 that reads```pythonprint(f"   {k:<12} -> {v!r}")```means: print `k` padded with spaces to 12 characters wide so the arrows line up in a column,then a literal ` -> `, then `v` **with its quotation marks shown**. The `!r` is theredeliberately — these values are label strings fed to a model, and seeing the quotes makesstray spaces or empty strings visible instead of invisible.

In [ ]:
def preprocess(text):    "Clean transcript text and split it into sentences."    # REGEX 1:  r"\s+"  =  one or more whitespace characters.    #   \s = any whitespace (space, tab, newline);  + = one or more of them.    # Replacing every such run with a single space turns "a\n\n   b" into "a b".    # We do this FIRST because stage directions can span lines in the source, and    # regex 2 below cannot match across a newline unless we collapse them now.    text = re.sub(r"\s+", " ", text).strip()    # REGEX 2:  r"\[.*?\]"  =  a [ , then as few characters as possible, then a ] .    #   \[ and \] = LITERAL square brackets (the backslash cancels their special meaning)    #   .   = any character    #   *?  = zero or more, LAZY - stop at the first ] rather than the last one    # Lazy matters: on "[Applause] and [Laughter]" a greedy .* would match the whole    # string and delete " and " too. The lazy version removes the two brackets only.    text = re.sub(r"\[.*?\]", " ", text)    sentences = sent_tokenize(text)    # Drop fragments: "Amen." is a real sentence, a stray "]" is not.    return [s.strip() for s in sentences if len(s.strip().split()) >= 3]speeches = {label: preprocess(raw) for label, raw in speeches_raw.items()}n_sent = sum(len(s) for s in speeches.values())print(f"{n_sent:,} sentences across {len(speeches)} speeches")print(f"Mean sentences per speech: {n_sent / len(speeches):.0f}\n")print("Three sentences from Kennedy 1961:")for s in speeches["Kennedy 1961"][:3]:    print("  •", s)

### 1.3 Optional: add your own speechesAnything after 2021 is not in NLTK. Drop `.txt` or `.rtf` files next to the notebook (Colab:folder icon → upload) and list them here. Leave the dictionary empty to skip.

In [ ]:
EXTRA_SPEECHES = {    # "Trump 2025": "trump2.rtf",}def load_text(path):    raw = Path(path).read_text(encoding="utf-8", errors="ignore")    if raw.lstrip().startswith("{\\rtf"):        from striprtf.striprtf import rtf_to_text        return rtf_to_text(raw)    return rawfor label, fname in EXTRA_SPEECHES.items():    if Path(fname).exists():        speeches[label] = preprocess(load_text(fname))        print(f"Added {label}: {len(speeches[label])} sentences")    else:        print(f"!! {fname} not found - skipping {label}")

### 1.4 The point of the whole design: sentence lengthsBefore we touch a model, look at how long these sentences are **in BERT tokens**. This plot isthe justification for the entire lab.

In [ ]:
tok_probe = AutoTokenizer.from_pretrained("bert-base-uncased")all_sentences = [s for sents in speeches.values() for s in sents]lengths = np.array([len(tok_probe.encode(s)) for s in all_sentences])print(f"median {np.median(lengths):.0f} tokens | "      f"90th pct {np.percentile(lengths, 90):.0f} | "      f"99th pct {np.percentile(lengths, 99):.0f} | max {lengths.max()}")print(f"Share under 64 tokens:  {(lengths <= 64).mean():.1%}")print(f"Share under 128 tokens: {(lengths <= 128).mean():.1%}")fig, ax = plt.subplots(figsize=(8, 4))ax.hist(lengths, bins=60, color="#0072B2", alpha=0.85)ax.axvline(64,  color="#D55E00", lw=2, label="MAX_LEN = 64 (our choice)")ax.axvline(512, color="#333333", lw=2, ls="--", label="BERT's hard limit = 512")ax.set_xlabel("sentence length (BERT tokens)")ax.set_ylabel("number of sentences")ax.set_title("Inaugural-address sentences fit inside BERT with room to spare")ax.legend(frameon=False)ax.spines[["top", "right"]].set_visible(False)plt.tight_layout(); plt.show()

**Read this before moving on.** Almost every sentence fits in 64 tokens — an eighth of whatBERT allows. That has three consequences:1. **No truncation**, so no silent data loss and no validity caveat to explain away.2. **Fast training.** Compute in a transformer scales with sequence length; 64 tokens instead   of 512 makes fine-tuning roughly an order of magnitude cheaper. This is why the whole lab   runs in minutes.3. **The unit of analysis is the unit of measurement.** Winter codes sentences; we classify   sentences. Nothing has to be aggregated, chunked, or approximated before the model sees it.Contrast this with applying BERT to full parliamentary speeches, where you must chunk ortruncate before you start, and every result carries an asterisk.

---# Part 1 — Zero-shot: classification with no training data## 2. How zero-shot classification worksA model trained on **natural language inference** (NLI) judges whether a premise *entails* ahypothesis. Zero-shot classification is a trick built on top of that: turn each candidate labelinto a hypothesis and ask which one the text entails.> premise: *"Let every nation know we shall pay any price to assure the survival of liberty."*> hypothesis: *"This example is about power and influence over others."* → entailment?Because the labels are supplied at prediction time, you can classify into any categories youlike without training anything. The cost is that **the label strings are model input** — howyou phrase a category changes the answer. That is prompt engineering wearing a lab coat, andwe test it directly in §3.2.

In [ ]:
zero_shot = pipeline("zero-shot-classification",                     model="roberta-large-mnli",                     device=0 if DEVICE == "cuda" else -1)demo = "We shall pay any price, bear any burden, to assure the survival of liberty."out = zero_shot(demo, ["power", "achievement", "affiliation"], multi_label=True)for lab, score in zip(out["labels"], out["scores"]):    print(f"  {lab:<12} {score:.3f}")

Note `multi_label=True`. With `False`, the scores are softmaxed and forced to sum to 1 —appropriate when exactly one label is correct. Motives are not mutually exclusive, so eachlabel gets its own independent probability. **This same choice reappears in §4.2 as sigmoidversus softmax; it is the single most important design decision in the lab.**## 3. Zero-shot motive scoring### 3.1 Writing the label setOur first attempt uses bare motive names. Winter's constructs are more specific than theeveryday words, so we should not expect much.

In [ ]:
LABEL_SETS = {    "bare": {        "achievement": "achievement",        "affiliation": "affiliation",        "power":       "power",    },    "winter": {        "achievement": "meeting a standard of excellence, doing something well or better",        "affiliation": "warm friendly feeling toward other people, companionship, belonging together",        "power":       "having impact on other people: influencing, persuading, controlling, or commanding them",    },}for name, mapping in LABEL_SETS.items():    print(f"[{name}]")    for k, v in mapping.items():        # f-string format specs (see the note in §1.2):        #   {k:<12}  left-align k in a column 12 characters wide, so the arrows line up        #   {v!r}    print repr(v) rather than str(v) - i.e. WITH its quote marks, so a        #            stray trailing space or an empty label string is visible, not silent        print(f"   {k:<12} -> {v!r}")

### 3.2 Does label wording matter?Run both label sets over the same handful of sentences and compare. This is the experimentworth remembering from Part 1.

In [ ]:
probe_sentences = [    "Let every nation know that we shall pay any price to assure the survival of liberty.",    "My fellow citizens, we are bound together in friendship and common purpose.",    "We will build a nation stronger and more prosperous than any that came before.",    "The oath I have taken before you today is the same oath taken by every president.",]rows = []for set_name, mapping in LABEL_SETS.items():    prompts = list(mapping.values())    prompt_to_motive = {v: k for k, v in mapping.items()}    for sent in probe_sentences:        res = zero_shot(sent, prompts, multi_label=True)        scores = {prompt_to_motive[l]: s for l, s in zip(res["labels"], res["scores"])}        rows.append({"label_set": set_name, "sentence": sent[:48] + "...", **scores})comparison = pd.DataFrame(rows).set_index(["sentence", "label_set"]).sort_index()comparison.round(3)

Look down each pair of rows. The *ranking* of motives often flips between label sets for theidentical sentence, and the absolute scores move a great deal. Nothing about the model changed —only the words we used to describe the categories.This is the central weakness of zero-shot for measurement: **you have no way to tell whetherthe model is measuring Winter's construct or your paraphrase of it**, and no labelled data withwhich to find out. That is the gap Part 2 closes.### 3.3 Zero-shot profiles for a few presidentsLet us get a baseline anyway. We score a random sample of sentences per speech (zero-shot isslow: every sentence needs one forward pass *per label*).

In [ ]:
SUBSET = ["Kennedy 1961", "Reagan 1981", "Obama 2009", "Trump 2017", "Biden 2021"]N_PER_SPEECH = 40mapping = LABEL_SETS["winter"]prompts = list(mapping.values())prompt_to_motive = {v: k for k, v in mapping.items()}zs_rows = []for label in SUBSET:    sents = speeches[label]    take = rng.choice(len(sents), size=min(N_PER_SPEECH, len(sents)), replace=False)    for i in tqdm(take, desc=label, leave=False):        res = zero_shot(sents[i], prompts, multi_label=True)        scores = {prompt_to_motive[l]: s for l, s in zip(res["labels"], res["scores"])}        zs_rows.append({"speech": label, **scores})zs = pd.DataFrame(zs_rows)zs_profile = zs.groupby("speech")[MOTIVES].mean().loc[SUBSET]print("Zero-shot motive profiles (mean score per sentence)\n")zs_profile.round(3)

Keep this table. In §7 we put it side by side with the fine-tuned model's profiles and withWinter's published human coding, and ask which one a reviewer should believe.

---# Part 2 — Fine-tuning on Winter's coded sentences## 4. Getting labelled data### 4.1 The Winter (1994) benchmarkThe sentences in Winter's *Manual for Scoring Motive Imagery in Running Text* have beendigitised: roughly 1,350 sentences, each coded for achievement, affiliation and power. It isthe set that human coders must reach >0.85 agreement with before they are considered certified,which makes it about as close to a gold standard as this field has.It is posted on OSF at **https://osf.io/6fnz5**.The cell below tries three sources in order and tells you which one it used:1. **OSF download** — the real thing.2. **A local CSV** you supply, if the automatic download fails (OSF sometimes requires a   browser). Download it by hand, upload it to the notebook, and set `LOCAL_CSV`.3. **Teacher-model labels** — a fallback that always works. A published motive classifier   labels our inaugural sentences, and we fine-tune on *those* labels. This is   knowledge distillation, and it is a legitimate technique, but the labels are a model's   output rather than Winter's coding. The notebook flags loudly when this path is taken.

In [ ]:
LOCAL_CSV = None          # e.g. "winter_benchmark.csv" if you downloaded it by handOSF_URL   = "https://osf.io/6fnz5/download"_ALIASES = {    "achievement": ["ach", "achievement", "nach", "achieve", "achievementimagery"],    "affiliation": ["aff", "affiliation", "naff", "affil", "intimacy",                    "affiliationintimacy"],    "power":       ["pow", "power", "npow", "pwr", "powerimagery"],}_TEXT_ALIASES = ["sentence", "text", "sentences", "item", "string", "content",                 "body", "utterance", "phrase"]def _norm(s):    # Reduce a column name to bare lowercase alphanumerics, for fuzzy matching.    #    # REGEX 3:  r"[^a-z0-9]"  =  any single character that is NOT a-z and NOT 0-9.    #   [ ]   = a character CLASS: match any ONE character from the set inside    #   ^     = at the START of a class it NEGATES the set: "anything except these"    #   a-z   = a range, the lowercase letters;   0-9 = a range, the digits    # Replacing every such character with the empty string strips spaces, underscores,    # dots and hyphens, so "n_Ach", "n.ach" and "N Ach" all collapse to "nach" and    # therefore compare equal. That is what lets the loader cope with an unknown schema.    return re.sub(r"[^a-z0-9]", "", str(s).lower())def find_columns(df):    "Locate the text column and one column per motive, whatever they happen to be called."    norm = {c: _norm(c) for c in df.columns}    text_aliases = [_norm(a) for a in _TEXT_ALIASES]    text_col = next((c for c, n in norm.items() if n in text_aliases), None)    if text_col is None:                       # fallback: the wordiest string column        strcols = [c for c in df.columns                   if df[c].dtype == object or pd.api.types.is_string_dtype(df[c])]        text_col = max(strcols, key=lambda c: df[c].astype(str).str.len().mean(), default=None)    labels = {}    for motive, aliases in _ALIASES.items():        aliases_n = [_norm(a) for a in aliases]        for c, n in norm.items():            if c == text_col:                continue            if n in aliases_n or any(n.startswith(a) for a in aliases_n):                labels[motive] = c                break    return text_col, labelsdef load_winter_benchmark():    "Return (DataFrame[text, achievement, affiliation, power], source_string)."    for source, path in [("OSF", OSF_URL), ("local CSV", LOCAL_CSV)]:        if path is None:            continue        for reader in (pd.read_csv, pd.read_excel):            try:                raw = reader(path)            except Exception:                continue            text_col, label_cols = find_columns(raw)            if text_col is None or set(label_cols) != set(MOTIVES):                continue            out = pd.DataFrame({"text": raw[text_col].astype(str)})            for m in MOTIVES:                out[m] = (pd.to_numeric(raw[label_cols[m]], errors="coerce")                            .fillna(0) > 0).astype(int)            out = out[out["text"].str.split().str.len() >= 3].reset_index(drop=True)            return out, source    return None, Nonewinter, SOURCE = load_winter_benchmark()if winter is not None:    print(f"Loaded {len(winter):,} coded sentences from: {SOURCE}")    print("\nPositive rate per motive:")    print(winter[MOTIVES].mean().round(3).to_string())else:    print("Could not reach the benchmark automatically.")    print("Run the next cell to use teacher-model labels instead,")    print("or download https://osf.io/6fnz5 by hand and set LOCAL_CSV above.")

In [ ]:
# FALLBACK ONLY - runs if the benchmark could not be loaded above.if winter is None:    print("=" * 72)    print("FALLBACK: labels come from a published classifier, NOT from Winter's coding.")    print("Everything below still demonstrates fine-tuning, but the 'gold standard'")    print("is another model's output. Treat §7's validation accordingly.")    print("=" * 72)    teacher = pipeline("text-classification",                       model="encodingai/electra-base-discriminator-im-multilabel-V3",                       top_k=None, device=0 if DEVICE == "cuda" else -1)    TEACHER_MAP = {"LABEL_0": "power", "LABEL_1": "achievement", "LABEL_2": "affiliation",                   "power": "power", "achievement": "achievement", "affiliation": "affiliation"}    pool = rng.choice(all_sentences, size=min(1500, len(all_sentences)), replace=False)    rows = []    for i in tqdm(range(0, len(pool), 32), desc="teacher labelling"):        batch = list(pool[i:i + 32])        for sent, preds in zip(batch, teacher(batch, truncation=True, max_length=64)):            scores = {TEACHER_MAP.get(p["label"], p["label"]): p["score"] for p in preds}            rows.append({"text": sent,                         **{m: int(scores.get(m, 0) >= 0.5) for m in MOTIVES}})    winter = pd.DataFrame(rows)    SOURCE = "teacher model (fallback)"    print(f"\nBuilt {len(winter):,} weakly-labelled sentences.")    print(winter[MOTIVES].mean().round(3).to_string())

In [ ]:
print(f"Training data source: {SOURCE}")print(f"Rows: {len(winter):,}\n")print("Label co-occurrence (how often motives appear together):")combos = winter[MOTIVES].astype(str).agg("".join, axis=1).value_counts()for pattern, n in combos.items():    active = [m for m, bit in zip(MOTIVES, pattern) if bit == "1"] or ["(none)"]    print(f"  {' + '.join(active):<40} {n:>5}  ({n/len(winter):.1%})")

### 4.2 Multi-label, not multi-classThat co-occurrence table is the reason this is not an ordinary classification problem. Sentencescarry two motives at once, and many carry none. Two consequences for the model:| | Multi-class (softmax) | **Multi-label (sigmoid)** ||---|---|---|| Assumption | exactly one label is correct | each label is independently present or absent || Output layer | softmax — scores sum to 1 | sigmoid — three independent probabilities || Loss | cross-entropy | binary cross-entropy, summed over labels || Labels look like | `2` | `[1, 0, 1]` |In `transformers` you get this by passing `problem_type="multi_label_classification"` andhanding the model **float** labels. Integer labels silently trigger the multi-class path, whichis one of the easiest ways to waste an afternoon.### 4.3 Tokenisation`MAX_LEN = 64`, justified by the histogram in §1.4. Padding to a fixed short length keepsbatches rectangular and small.

In [ ]:
from sklearn.model_selection import train_test_splitMODEL_NAME = "bert-base-uncased"MAX_LEN = 64tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)train_df, test_df = train_test_split(    winter, test_size=0.2, random_state=42,    stratify=winter["power"]           # keep the commonest motive balanced across splits)train_df = train_df.reset_index(drop=True)test_df  = test_df.reset_index(drop=True)print(f"Train: {len(train_df):,}   Test: {len(test_df):,}")def encode(df):    enc = tokenizer(list(df["text"]), truncation=True, padding="max_length",                    max_length=MAX_LEN, return_tensors="pt")    y = torch.tensor(df[MOTIVES].values, dtype=torch.float)   # FLOAT, see §4.2    return enc, ytrain_enc, train_y = encode(train_df)test_enc,  test_y  = encode(test_df)print(f"input_ids shape: {tuple(train_enc['input_ids'].shape)}  (examples x tokens)")print(f"labels shape:    {tuple(train_y.shape)}  (examples x motives)")ex = train_df["text"].iloc[0]print(f"\nExample: {ex[:90]}")print("Tokens :", tokenizer.convert_ids_to_tokens(tokenizer(ex)["input_ids"])[:20])

### 4.4 A Dataset and DataLoaderSmall enough to read in one sitting: it hands back one dictionary per example.

In [ ]:
from torch.utils.data import Dataset, DataLoaderclass MotiveDataset(Dataset):    def __init__(self, encodings, labels):        self.encodings, self.labels = encodings, labels    def __len__(self):        return len(self.labels)    def __getitem__(self, i):        item = {k: v[i] for k, v in self.encodings.items()}        item["labels"] = self.labels[i]        return itemtrain_ds = MotiveDataset(train_enc, train_y)test_ds  = MotiveDataset(test_enc,  test_y)train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)test_dl  = DataLoader(test_ds,  batch_size=64)print(f"{len(train_dl)} training batches of 32")

### 4.5 The training loopYour other notebook uses Hugging Face's `Trainer`, which is the right tool for real work. Herewe write the loop by hand, because the whole point of the lab is to *see* what fine-tuning does.It is nine lines:> zero the gradients → forward pass → compute loss → backpropagate → step the optimiser`BertForSequenceClassification` loads pretrained BERT and bolts on a **randomly initialised**classification head sized to `num_labels`. The warning about newly initialised weights isexpected and correct — that head is exactly what we are training.

In [ ]:
def build_model():    return AutoModelForSequenceClassification.from_pretrained(        MODEL_NAME,        num_labels=len(MOTIVES),        problem_type="multi_label_classification",   # -> sigmoid + BCE loss    ).to(DEVICE)def train(model, dataloader, epochs=3, lr=2e-5, verbose=True):    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)    model.train()    for epoch in range(epochs):        total = 0.0        for batch in dataloader:            batch = {k: v.to(DEVICE) for k, v in batch.items()}            optimizer.zero_grad()            out = model(**batch)            out.loss.backward()            optimizer.step()            total += out.loss.item()        if verbose:            print(f"  epoch {epoch + 1}/{epochs}  mean loss {total / len(dataloader):.4f}")    return modelmodel = build_model()print(f"Fine-tuning {MODEL_NAME} on {len(train_ds):,} sentences...")train(model, train_dl, epochs=3)

### 4.6 EvaluationPredictions come out as logits. We apply a sigmoid to get one probability per motive, thenthreshold at 0.5. Because the labels are independent, we report precision/recall/F1 **permotive**.

In [ ]:
from sklearn.metrics import classification_report, f1_score@torch.no_grad()def predict_proba(model, dataloader):    model.eval()    out = []    for batch in dataloader:        batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}        out.append(torch.sigmoid(model(**batch).logits).cpu().numpy())    return np.vstack(out)probs = predict_proba(model, test_dl)preds = (probs >= 0.5).astype(int)gold  = test_y.numpy().astype(int)print(classification_report(gold, preds, target_names=MOTIVES, zero_division=0))print(f"Macro F1: {f1_score(gold, preds, average='macro', zero_division=0):.3f}")

### 4.7 How much labelled data do you actually need?The question every project faces. We retrain from scratch on progressively larger subsets andplot the curve.**Watch what happens at n = 10.** Fine-tuning 110 million parameters on ten examples produces amodel that predicts the majority class for everything and scores an F1 near zero. That is not abug, and it is worth sitting with: few-shot fine-tuning of a full encoder does not work. If yougenuinely have ten labels, you need a different method (zero-shot, or a SetFit-stylesentence-embedding classifier), not a smaller learning rate.

In [ ]:
SIZES = [10, 25, 50, 100, 250, 500, len(train_ds)]SIZES = sorted({min(n, len(train_ds)) for n in SIZES})curve = []for n in SIZES:    idx = rng.choice(len(train_ds), size=n, replace=False)    sub_enc = {k: v[idx] for k, v in train_enc.items()}    sub_dl  = DataLoader(MotiveDataset(sub_enc, train_y[idx]),                         batch_size=min(32, n), shuffle=True)    m = build_model()    train(m, sub_dl, epochs=3, verbose=False)    p = (predict_proba(m, test_dl) >= 0.5).astype(int)    macro = f1_score(gold, p, average="macro", zero_division=0)    curve.append({"n": n, "macro_f1": macro,                  **{mot: f1_score(gold[:, j], p[:, j], zero_division=0)                     for j, mot in enumerate(MOTIVES)}})    print(f"  n = {n:>5}   macro F1 = {macro:.3f}")    del m    if DEVICE == "cuda":        torch.cuda.empty_cache()curve_df = pd.DataFrame(curve)curve_df.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))ax.plot(curve_df["n"], curve_df["macro_f1"], "o-", lw=2.5,        color="#0072B2", label="macro F1", zorder=3)for mot, c in zip(MOTIVES, ["#E69F00", "#009E73", "#D55E00"]):    ax.plot(curve_df["n"], curve_df[mot], "o--", lw=1.2, alpha=0.75, color=c, label=mot)ax.set_xscale("log")ax.set_xlabel("labelled training sentences (log scale)")ax.set_ylabel("F1 on held-out test set")ax.set_title("How much labelled data does fine-tuning need?")ax.axhline(0, color="#999999", lw=0.8)ax.legend(frameon=False)ax.spines[["top", "right"]].set_visible(False)ax.grid(True, alpha=0.3, lw=0.4)plt.tight_layout(); plt.show()

**Questions for the room:**- Where does the curve flatten? That elbow is your labelling budget.- Which motive is learned fastest, and why might that be? (Look back at the positive rates in  §4.1 — rare categories need more data.)- At the far left, the model is worse than useless. At what n does it beat the zero-shot  baseline from §3.3, which needed no labels at all?

---## 5. Applying the model to two centuries of inauguralsNow the measurement. We score every sentence in all 59 addresses and aggregate to a motiveprofile per president: **the proportion of sentences carrying each motive** — which is, inspirit, exactly what Winter's hand-coded scores are.

In [ ]:
@torch.no_grad()def score_sentences(model, sentences, batch_size=128):    model.eval()    out = []    for i in range(0, len(sentences), batch_size):        enc = tokenizer(sentences[i:i + batch_size], truncation=True,                        padding="max_length", max_length=MAX_LEN,                        return_tensors="pt").to(DEVICE)        out.append(torch.sigmoid(model(**enc).logits).cpu().numpy())    return np.vstack(out)records = []for label, sents in tqdm(speeches.items(), desc="scoring speeches"):    p = score_sentences(model, sents)    year = int(label.split()[-1])    records.append({"speech": label, "president": label.rsplit(" ", 1)[0],                    "year": year, "n_sentences": len(sents),                    **{m: (p[:, j] >= 0.5).mean() for j, m in enumerate(MOTIVES)}})profiles = pd.DataFrame(records).sort_values("year").reset_index(drop=True)print(f"Scored {profiles['n_sentences'].sum():,} sentences.\n")# Sanity check: a model that never fires on a motive makes a flat, useless plot.for m in MOTIVES:    if profiles[m].max() == 0:        print(f"!! WARNING: '{m}' was never predicted anywhere in the corpus.")        print(f"   The model collapsed on this label - check its F1 in §4.6 and the")        print(f"   positive rate in §4.1. Lowering the threshold below 0.5 may help,")        print(f"   but a collapsed label usually means too little training signal.\n")profiles.tail(10).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))for mot, c in zip(MOTIVES, ["#E69F00", "#009E73", "#D55E00"]):    ax.plot(profiles["year"], profiles[mot], "o-", ms=4, lw=1.4,            alpha=0.85, color=c, label=mot)    ax.plot(profiles["year"], profiles[mot].rolling(5, center=True, min_periods=1).mean(),            lw=3, alpha=0.35, color=c)ax.set_xlabel("year of inaugural address")ax.set_ylabel("share of sentences with motive imagery")ax.set_title("Motive imagery in U.S. inaugural addresses, 1789-2021 (fine-tuned BERT)")ax.legend(frameon=False, ncol=3)ax.spines[["top", "right"]].set_visible(False)ax.grid(True, alpha=0.3, lw=0.4)plt.tight_layout(); plt.show()

In [ ]:
# Which sentences does the model consider most strongly power-motivated?probe = "Kennedy 1961"p = score_sentences(model, speeches[probe])print(f"Most POWER-imagery sentences in {probe}:\n")for i in np.argsort(-p[:, MOTIVES.index("power")])[:5]:    print(f"  [{p[i, MOTIVES.index('power')]:.2f}] {speeches[probe][i]}")print(f"\nMost AFFILIATION-imagery sentences in {probe}:\n")for i in np.argsort(-p[:, MOTIVES.index("affiliation")])[:5]:    print(f"  [{p[i, MOTIVES.index('affiliation')]:.2f}] {speeches[probe][i]}")

**Face validity is a real check.** Read those sentences. If the top "power" sentences are aboutstrength, command and influence, and the top "affiliation" sentences are about togetherness andshared feeling, the model is measuring something recognisable. If they look arbitrary, no F1score will save the measure.

---## 6. Follow-up: which presidents resemble each other?The fine-tuned model gives us two quite different ways to compare speeches, and the interestingresult comes from comparing the two against each other.1. **Motive space** — each speech is now a point in three dimensions: (achievement,   affiliation, power). This is Winter's construct, and nothing else.2. **Representation space** — the fine-tuned encoder's 768-dimensional hidden states, which   have been *reshaped* by training on the motive task.The coda in §9 does the same comparison with an encoder that was never fine-tuned. Holdingthose three side by side answers a question students always ask and rarely get an answer to:**what does fine-tuning actually change?**### 6.1 A data trap first: surnames are not peopleBefore comparing presidents we have to disambiguate them. NLTK names its files by surname, andfour surnames belong to two different men:| Surname | Person A | Person B ||---|---|---|| Adams | John Adams (1797) | John Quincy Adams (1825) || Harrison | William Henry Harrison (1841) | Benjamin Harrison (1889) || Roosevelt | Theodore Roosevelt (1905) | Franklin D. Roosevelt (1933-45) || Bush | George H. W. Bush (1989) | George W. Bush (2001, 2005) |Grover Cleveland, by contrast, really is one man with two non-consecutive inaugurals (1885 and1893), so he must **not** be split. Get this wrong and the same-speaker test in §6.3 is silentlycorrupted — a father and son get counted as one person, and the "stability of individual style"result becomes an artifact of file naming.

In [ ]:
# Surnames shared by two different presidents: split them at a year boundary.# Speeches BEFORE the cutoff are person A; speeches from the cutoff onward are person B.COLLIDING = {"Adams": 1820, "Harrison": 1880, "Roosevelt": 1930, "Bush": 2000}def person_id(surname, year):    "Turn (surname, year) into a unique person, splitting the four shared surnames."    if surname in COLLIDING:        return f"{surname}-{'I' if year < COLLIDING[surname] else 'II'}"    return surnameprofiles["person"] = [person_id(r.president, r.year) for r in profiles.itertuples()]counts = profiles["person"].value_counts()repeat = counts[counts > 1]print(f"{profiles['person'].nunique()} distinct presidents, "      f"{len(repeat)} of whom gave more than one inaugural.\n")print("Presidents with multiple inaugurals:")for person, n in repeat.items():    years = sorted(profiles.loc[profiles["person"] == person, "year"])    print(f"  {person:<14} {n} speeches  {years}")

### 6.2 Cosine similarity in motive spaceCosine similarity compares the *direction* of two vectors, ignoring their length:$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\lVert\mathbf{a}\rVert \, \lVert\mathbf{b}\rVert}$$Here that has a clean substantive reading. Length is *how motive-laden* a speech is overall;direction is *how the speaker divides their energy* between the three motives. Two presidentsscore 1.0 if they have the same motivational profile, even if one is far more emphatic than theother. That is usually what you want when comparing rhetoric across eras with differentstylistic conventions.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity# Each row is one speech; the three columns are the motive proportions from §5.motive_vecs = profiles[MOTIVES].to_numpy()# Guard: a speech with no motive imagery at all is the zero vector, and cosine# similarity is undefined for it (0/0). Drop those rather than emit silent NaNs.nonzero = motive_vecs.sum(axis=1) > 0if (~nonzero).any():    print(f"Dropping {(~nonzero).sum()} speech(es) with no predicted motive imagery.\n")prof_ok = profiles[nonzero].reset_index(drop=True)motive_sim = cosine_similarity(prof_ok[MOTIVES].to_numpy())speech_names = list(prof_ok["speech"])pair_ix = np.triu_indices_from(motive_sim, k=1)   # upper triangle, excluding the diagonalorder = np.argsort(-motive_sim[pair_ix])print("Most similar pairs in motive space:")for k in order[:6]:    i, j = pair_ix[0][k], pair_ix[1][k]    print(f"  {motive_sim[i, j]:.4f}   {speech_names[i]:<18} <-> {speech_names[j]}")print("\nLeast similar pairs in motive space:")for k in order[-6:][::-1]:    i, j = pair_ix[0][k], pair_ix[1][k]    print(f"  {motive_sim[i, j]:.4f}   {speech_names[i]:<18} <-> {speech_names[j]}")

In [ ]:
# A 59x59 heatmap is unreadable, so plot the most recent 16 speeches.RECENT_N = 16recent_idx = list(range(len(prof_ok)))[-RECENT_N:]sub = motive_sim[np.ix_(recent_idx, recent_idx)]sub_names = [speech_names[i] for i in recent_idx]fig, ax = plt.subplots(figsize=(9, 7.5))im = ax.imshow(sub, cmap="Blues", vmin=np.min(sub), vmax=1.0)ax.set_xticks(range(len(sub_names)), sub_names, rotation=45, ha="right", fontsize=8)ax.set_yticks(range(len(sub_names)), sub_names, fontsize=8)for i in range(len(sub_names)):    for j in range(len(sub_names)):        shade = (sub[i, j] - im.get_clim()[0]) / max(np.ptp(im.get_clim()), 1e-9)        ax.text(j, i, f"{sub[i, j]:.2f}", ha="center", va="center", fontsize=6.5,                color="white" if shade > 0.6 else "#1a1a1a")ax.set_title("Cosine similarity in motive space (fine-tuned BERT)", pad=12)fig.colorbar(im, ax=ax, shrink=0.8, label="cosine similarity")plt.tight_layout(); plt.show()

### 6.3 The same-speaker testHere is a validity check that costs nothing and needs no external data.Sixteen presidents gave more than one inaugural address. If our measure captures somethingstable about *the person* — rather than the year, the war, or the speechwriter — then twospeeches by the same president should be more similar to each other than two speeches picked atrandom.We test it with a **permutation test**: compute the observed gap between within-speaker andbetween-speaker similarity, then shuffle the president labels 999 times to build a nulldistribution of the gaps that could arise by chance alone.

In [ ]:
persons = prof_ok["person"].to_numpy()same = persons[:, None] == persons[None, :]        # boolean matrix: same president?pair_ix = np.triu_indices_from(motive_sim, k=1)    # each pair counted oncewithin  = motive_sim[pair_ix][same[pair_ix]]between = motive_sim[pair_ix][~same[pair_ix]]observed = within.mean() - between.mean()print(f"Within-speaker pairs : {len(within):>5}   mean similarity {within.mean():.4f}")print(f"Between-speaker pairs: {len(between):>5}   mean similarity {between.mean():.4f}")print(f"Observed gap         : {observed:+.4f}\n")# Permutation null: shuffle which speech belongs to which president.perm_rng = np.random.default_rng(42)null = np.empty(999)for b in range(999):    shuffled = perm_rng.permutation(persons)    s_mat = (shuffled[:, None] == shuffled[None, :])[pair_ix]    null[b] = motive_sim[pair_ix][s_mat].mean() - motive_sim[pair_ix][~s_mat].mean()pval = (np.sum(null >= observed) + 1) / (len(null) + 1)print(f"Permutation p (one-sided) = {pval:.3f}")print("Interpretation: a small p means presidents really do repeat their own motive profile;")print("                a large p means the measure is picking up era or occasion, not person.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))ax.hist(null, bins=40, color="#999999", alpha=0.8, label="null (labels shuffled)")ax.axvline(observed, color="#D55E00", lw=2.5, label=f"observed = {observed:+.4f}")ax.set_xlabel("mean within-speaker similarity minus mean between-speaker similarity")ax.set_ylabel("permutations")ax.set_title("Do presidents repeat their own motive profile?")ax.legend(frameon=False)ax.spines[["top", "right"]].set_visible(False)plt.tight_layout(); plt.show()

### 6.4 What did fine-tuning do to the geometry?Now the question this section was built for.We take the **same architecture** (`bert-base-uncased`) in two states — untouched, and fine-tunedon motive imagery — and build a document embedding from each by mean-pooling the last hiddenlayer over all of a speech's sentences. Same model, same tokenizer, same texts. The onlydifference is the gradient steps we ran in §4.5.Then we ask whether the two spaces agree about *who resembles whom*.

In [ ]:
@torch.no_grad()def document_embedding(encoder, sentences, batch_size=64, from_classifier=False):    "Mean-pool the last hidden layer over all sentences in a speech -> one vector."    vecs = []    for i in range(0, len(sentences), batch_size):        enc = tokenizer(sentences[i:i + batch_size], truncation=True,                        padding="max_length", max_length=MAX_LEN,                        return_tensors="pt").to(DEVICE)        if from_classifier:            # A *ForSequenceClassification model does not return hidden states by            # default - ask for them explicitly, then take the final layer.            hidden = encoder(**enc, output_hidden_states=True).hidden_states[-1]        else:            hidden = encoder(**enc).last_hidden_state        mask = enc["attention_mask"].unsqueeze(-1)        vecs.append(((hidden * mask).sum(1) / mask.sum(1).clamp(min=1)).cpu().numpy())    return np.vstack(vecs).mean(axis=0)        # speech vector = mean of its sentence vectors# The untouched encoder: the weights BERT shipped with, no fine-tuning at all.plain_bert = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()model.eval()tuned_docs, plain_docs = [], []for name in tqdm(speech_names, desc="embedding speeches"):    sents = speeches[name]    tuned_docs.append(document_embedding(model, sents, from_classifier=True))    plain_docs.append(document_embedding(plain_bert, sents))tuned_sim = cosine_similarity(np.vstack(tuned_docs))plain_sim = cosine_similarity(np.vstack(plain_docs))print(f"\nBuilt {len(speech_names)} document embeddings in each space.")

In [ ]:
# Mantel-style test: do two similarity matrices agree about the ordering of pairs?def matrix_correlation(A, B, n_perm=999, seed=42):    ix = np.triu_indices_from(A, k=1)    b = B[ix]    obs = np.corrcoef(A[ix], b)[0, 1]    g = np.random.default_rng(seed)    null = np.empty(n_perm)    for i in range(n_perm):                    # permute rows AND columns together,        p = g.permutation(len(A))              # which preserves the matrix structure        null[i] = np.corrcoef(A[np.ix_(p, p)][ix], b)[0, 1]    return obs, (np.sum(np.abs(null) >= abs(obs)) + 1) / (n_perm + 1)r_tp, p_tp = matrix_correlation(tuned_sim, plain_sim)print("Agreement between similarity spaces (Mantel-style permutation test)\n")print(f"  fine-tuned  vs  untouched BERT     r = {r_tp:+.3f}   p = {p_tp:.3f}\n")ix = np.triu_indices_from(motive_sim, k=1)r_tuned = np.corrcoef(motive_sim[ix], tuned_sim[ix])[0, 1]r_plain = np.corrcoef(motive_sim[ix], plain_sim[ix])[0, 1]print(f"  motive space vs FINE-TUNED  representation   r = {r_tuned:+.3f}")print(f"  motive space vs UNTOUCHED   representation   r = {r_plain:+.3f}")print(f"\n  Fine-tuning moved the geometry toward the construct by "      f"{r_tuned - r_plain:+.3f} in correlation.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=False, sharey=True)ix = np.triu_indices_from(motive_sim, k=1)for ax, (title, mat) in zip(axes, [("untouched bert-base", plain_sim),                                   ("fine-tuned on motives", tuned_sim)]):    ax.scatter(mat[ix], motive_sim[ix], s=8, alpha=0.25, color="#0072B2", edgecolors="none")    r = np.corrcoef(mat[ix], motive_sim[ix])[0, 1]    ax.set_title(f"{title}\n(r with motive space = {r:+.3f})", fontsize=10)    ax.set_xlabel("cosine similarity in representation space")    ax.spines[["top", "right"]].set_visible(False)    ax.grid(True, alpha=0.3, lw=0.4)axes[0].set_ylabel("cosine similarity in motive space")fig.suptitle("Does the encoder's geometry line up with the motive measure?", y=1.02)plt.tight_layout(); plt.show()

**How to read this.** Each dot is one pair of speeches.If the right-hand panel shows a tighter relationship than the left, fine-tuning has pulled theencoder's notion of similarity *toward* the motive construct — the representation now organisesspeeches by motivational profile rather than by topic, era, or style. That is what fine-tuningis supposed to do, and here you can watch it happen rather than take it on faith.Two cautions to raise in the room:- **Untouched BERT similarities are anisotropic.** Raw encoder embeddings occupy a narrow cone,  so every pair looks similar (often > 0.95) and the left-hand panel is compressed into a  narrow strip. Read the *ordering* of pairs, not the absolute values — the same warning as §9.- **Agreement is not correctness.** The fine-tuned representation agreeing with the motive  scores partly just reflects that the same model produced both. The independent check is still  §7: correlation with Winter's human coding.

---## 7. Validation: the part that mattersA held-out F1 tells you the model reproduces its own training distribution. It does **not** tellyou the model measures presidential motives. For that you need a criterion measuredindependently — and here we have two.### 7.1 Against Winter's published presidential scoresWinter published motive scores for U.S. presidents from hand-coded inaugural addresses. Fill inthe table below from the source you are teaching (Winter 1987, *JPSP*, is the standard citation;later papers extend the series), then run the correlation.Winter's published scores are standardised as **images per 1,000 words**, so they are not on thesame scale as our proportions. Correlation is the right comparison, not agreement.

In [ ]:
# Enter Winter's published scores here (images per 1,000 words).# Leave as NaN for presidents you have not entered; they are dropped from the correlation.winter_published = pd.DataFrame({    "speech":      ["Kennedy 1961", "Johnson 1965", "Nixon 1969", "Carter 1977",                    "Reagan 1981", "Bush 1989", "Clinton 1993", "Obama 2009"],    "w_achievement": [np.nan] * 8,     # <- fill in    "w_affiliation": [np.nan] * 8,     # <- fill in    "w_power":       [np.nan] * 8,     # <- fill in})merged = profiles.merge(winter_published, on="speech", how="inner")usable = merged.dropna(subset=["w_achievement", "w_affiliation", "w_power"], how="all")if len(usable) >= 3:    print(f"Comparing {len(usable)} speeches\n")    for m in MOTIVES:        pair = usable[[m, f"w_{m}"]].dropna()        if len(pair) < 3:            print(f"  {m:<12} not enough published scores entered (n = {len(pair)})")        elif pair[m].nunique() < 2 or pair[f"w_{m}"].nunique() < 2:            print(f"  {m:<12} no variance to correlate "                  f"(model sd = {pair[m].std():.4f}, Winter sd = {pair[f'w_{m}'].std():.4f})")        else:            print(f"  {m:<12} r = {pair[m].corr(pair[f'w_{m}']):+.3f}  (n = {len(pair)})")else:    print("No published scores entered yet.")    print("Fill in winter_published above, then re-run this cell.")    print("\nOur model's estimates for those speeches:")    print(merged[["speech"] + MOTIVES].round(3).to_string(index=False))

### 7.2 Against a published motive classifierSomeone has already done this properly. `encodingai/electra-base-discriminator-im-multilabel-V3`is an ELECTRA model fine-tuned for implicit motive coding, reported at ICC ≈ 0.91 againstWinter's benchmark. Comparing our lab-grade model against it is cheap and informative.**This is the PopBERT lesson from the day-2 transformers notebook, made concrete:** beforetraining your own model, check whether a validated one already exists.

In [ ]:
try:    reference = pipeline("text-classification",                         model="encodingai/electra-base-discriminator-im-multilabel-V3",                         top_k=None, device=0 if DEVICE == "cuda" else -1)    REF_MAP = {"LABEL_0": "power", "LABEL_1": "achievement", "LABEL_2": "affiliation",               "power": "power", "achievement": "achievement", "affiliation": "affiliation"}    check = list(rng.choice(all_sentences, size=300, replace=False))    ref_scores = []    for i in range(0, len(check), 32):        batch = check[i:i + 32]        for preds in reference(batch, truncation=True, max_length=64):            s = {REF_MAP.get(p["label"], p["label"]): p["score"] for p in preds}            ref_scores.append([s.get(m, np.nan) for m in MOTIVES])    ref_scores = np.array(ref_scores)    our_scores = score_sentences(model, check)    print("Sentence-level correlation with the published ELECTRA coder (n = 300):\n")    for j, m in enumerate(MOTIVES):        r = np.corrcoef(our_scores[:, j], ref_scores[:, j])[0, 1]        agree = ((our_scores[:, j] >= .5) == (ref_scores[:, j] >= .5)).mean()        print(f"  {m:<12} r = {r:+.3f}   binary agreement = {agree:.1%}")except Exception as e:    print("Could not load the reference model:", type(e).__name__, str(e)[:160])

### 7.3 Lining up all three methodsThe comparison table your course keeps returning to.

In [ ]:
summary = pd.DataFrame({    "method": ["Zero-shot RoBERTa-MNLI", "Fine-tuned BERT (this lab)",               "Published ELECTRA coder"],    "labelled data": ["none", f"{len(train_ds):,} sentences", "none (already trained)"],    "compute": ["inference only, slow", "~1 min on a T4", "inference only"],    "validated against Winter": ["no", "held-out split", "ICC = 0.91 (published)"],})print(summary.to_string(index=False))

**Worth arguing about:**- Zero-shot needs no labels. Fine-tuning needs about a thousand. If the fine-tuned model reaches  only modestly higher F1, when is labelling worth it?- Our model was trained on sentences from a 1994 coding manual and applied to speeches from 1789.  Which direction does that bias run, and how would you detect it?- The ELECTRA model is better than ours and free. What is the argument for training your own  anyway? (There is one — it involves knowing exactly what your training data contained.)

---## 8. Exercises**1. Label wording (easy).** Go back to §3.1 and add a third label set, phrased in your ownwords from Winter's category definitions in the introduction. Does zero-shot improve? Compareagainst the fine-tuned model's predictions on the same sentences.**2. Threshold choice (easy).** We threshold sigmoid output at 0.5 throughout. Sweep thethreshold from 0.1 to 0.9 and plot precision against recall for each motive. Is 0.5 the rightchoice? Should it be the same for all three motives?**3. Freezing layers (medium).** With ~1,000 training examples, fine-tuning all 110M parametersinvites overfitting. Freeze everything except the top four encoder layers and the classifier,retrain, and compare. Does it help, and does it train faster?```pythonfor name, param in model.named_parameters():    if not any(name.startswith(p) for p in               ["bert.encoder.layer.8", "bert.encoder.layer.9",                "bert.encoder.layer.10", "bert.encoder.layer.11", "classifier"]):        param.requires_grad = False```**4. Error analysis (medium).** Pull the test-set sentences where the model is most confidentlywrong. Are they genuinely mislabelled by the model, or is the *gold* label debatable? Winter'smanual exists because these judgements are hard.**5. Aggregation (harder).** We aggregate by proportion of sentences. Winter uses images per1,000 words. Recompute the profiles Winter's way and see whether the presidential rank orderingchanges. Which aggregation would you defend in a paper?**6. Beyond inaugurals (harder).** The model is trained on manual sentences and applied toinaugural addresses — a domain shift. Apply it to a different genre (State of the Unionaddresses, `nltk.corpus.state_union`) and inspect whether the top-scoring sentences still lookright. Cross-domain validity is where most measurement papers get their referee reports.

---## 9. Coda: embeddings without fine-tuningThe earlier version of this notebook compared speeches using *unsupervised* RoBERTa embeddingsplus PCA and cosine similarity. That material is not obsolete — it answers a different question.Fine-tuning asks *"how much power imagery is in this speech?"*; embeddings ask *"which speechesresemble each other?"* with no target concept at all.The cell below reproduces it compactly for a handful of speeches, so you can hold the twoapproaches side by side.

In [ ]:
from sklearn.decomposition import PCAfrom sklearn.metrics.pairwise import cosine_similarityCODA = ["Kennedy 1961", "Reagan 1981", "Obama 2009", "Trump 2017", "Biden 2021"]enc_model = AutoModel.from_pretrained("roberta-base").to(DEVICE).eval()enc_tok   = AutoTokenizer.from_pretrained("roberta-base")@torch.no_grad()def embed(sentences, batch_size=32):    out = []    for i in range(0, len(sentences), batch_size):        e = enc_tok(sentences[i:i + batch_size], padding=True, truncation=True,                    max_length=MAX_LEN, return_tensors="pt").to(DEVICE)        hidden = enc_model(**e).last_hidden_state        mask = e["attention_mask"].unsqueeze(-1)             # mean-pool, ignoring padding        out.append(((hidden * mask).sum(1) / mask.sum(1).clamp(min=1)).cpu().numpy())    return np.vstack(out)sent_emb = {lab: embed(speeches[lab]) for lab in CODA}doc_emb  = np.vstack([e.mean(axis=0) for e in sent_emb.values()])sim = cosine_similarity(doc_emb)print("Cosine similarity between speeches (read the ORDERING, not the magnitudes -")print("raw encoder embeddings are anisotropic, so everything looks similar):\n")print(pd.DataFrame(sim, index=CODA, columns=CODA).round(3).to_string())

In [ ]:
X = np.vstack(list(sent_emb.values()))owner = np.concatenate([[lab] * len(e) for lab, e in sent_emb.items()])pca = PCA(n_components=2, random_state=42)X2 = pca.fit_transform(X)D2 = pca.transform(doc_emb)evr = pca.explained_variance_ratio_PALETTE = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7"]fig, ax = plt.subplots(figsize=(9, 6))for i, lab in enumerate(CODA):    rows = owner == lab    ax.scatter(X2[rows, 0], X2[rows, 1], s=18, alpha=0.4,               color=PALETTE[i % len(PALETTE)], label=lab, edgecolors="none")    ax.scatter(*D2[i], s=260, marker="*", color=PALETTE[i % len(PALETTE)],               edgecolors="black", linewidths=0.8, zorder=5)    ax.annotate(lab, D2[i], xytext=(8, 6), textcoords="offset points",                fontsize=9, fontweight="bold")ax.set_xlabel(f"PC1 ({evr[0]:.1%})")ax.set_ylabel(f"PC2 ({evr[1]:.1%})")ax.set_title("Unsupervised sentence embeddings (dots) and speech means (stars)")ax.legend(frameon=False, fontsize=8)ax.spines[["top", "right"]].set_visible(False)plt.tight_layout(); plt.show()

---### What to take away1. **Match the unit of analysis to the model's context window.** This lab is fast and clean   because Winter's method and BERT's architecture happen to agree that the sentence is the   unit. When they disagree, everything gets harder.2. **Zero-shot is free and unfalsifiable.** With no labelled data you also have no way to know   what you measured.3. **Fine-tuning needs hundreds, not tens, of examples** — §4.7 shows exactly where the curve   turns.4. **Held-out F1 is not validation.** Correlating with Winter's independently coded scores is.5. **Check for an existing model first.** Ours is a teaching artifact; the published ELECTRA   coder is what you would actually use in a paper.### References- Winter, D. G. (1994). *Manual for scoring motive imagery in running text* (4th ed.).  University of Michigan. https://deepblue.lib.umich.edu/handle/2027.42/117563- Winter, D. G. (1987). Leader appeal, leader performance, and the motive profiles of leaders  and followers. *Journal of Personality and Social Psychology, 52*(1), 196-202.- Benchmark data: https://osf.io/6fnz5- Reference model: https://huggingface.co/encodingai/electra-base-discriminator-im-multilabel-V3